# Phase B0 — Server-side batch-settlement spike

Confirms the exact per-request behavior of a **resource-server**-side `BatchSettlementEvmScheme`
(the seller/merchant role — this is what `scw_js` needs, NOT the buyer role the
`x402_facilitator/notebooks/x402_batch_settlement_buyer.ipynb` spike covers) before writing the real
S3-backed handler (`sc_llm_x402.ts`, Phase B3). See `assistent_plan.md` §F Phase B.

**Prerequisites**
- Deno Jupyter kernel.
- `scw_js/.env` with `NFT_WALLET_PUBLIC_KEY` (reused as the receiver address — no new wallet needed for this spike).
- `x402_facilitator/.env` with a funded `TEST_WALLET_PRIVATE_KEY` (same wallet that produced Phase A's real
  on-chain proofs) — reused here as the buyer, so we can observe the *success* path, not just failure.
- A facilitator running locally: from `x402_facilitator/`, `npm install && npm run build && npm run dev` → `http://localhost:8080`.

> ⚠️ This notebook submits a **real on-chain deposit** on Base Sepolia (Step 6). Costs real (tiny) testnet gas.


In [ ]:
import { load } from "https://deno.land/std@0.224.0/dotenv/mod.ts";
import { privateKeyToAccount, generatePrivateKey } from "npm:viem@2/accounts";
import { HTTPFacilitatorClient, x402ResourceServer } from "npm:@x402/core@2.17.0/server";
import {
    BatchSettlementEvmScheme as ServerScheme,
    InMemoryChannelStorage,
} from "npm:@x402/evm@2.17.0/batch-settlement/server";
import {
    BatchSettlementEvmScheme as ClientScheme,
    InMemoryClientChannelStorage,
    signVoucher,
} from "npm:@x402/evm@2.17.0/batch-settlement/client";

// scw_js's own env (receiver address) — the single source of truth, one level up.
const env = await load({ envPath: "../.env", examplePath: null, export: true });
// Reuse the funded Base Sepolia test wallet from the sibling x402_facilitator package
// (same one that produced Phase A's real on-chain proofs), purely for this spike.
const facEnv = await load({ envPath: "../../x402_facilitator/.env", examplePath: null, export: false });

const RECEIVER = env.NFT_WALLET_PUBLIC_KEY as `0x${string}`;
const network = "eip155:84532"; // Base Sepolia
const FACILITATOR_URL = "http://localhost:8080";

const buyerAccount = privateKeyToAccount(`0x${facEnv.TEST_WALLET_PRIVATE_KEY.replace(/^0x/, "")}`);
const buyerSigner = { address: buyerAccount.address, signTypedData: (a: any) => buyerAccount.signTypedData(a) };
const buyerScheme = new ClientScheme(buyerSigner, { storage: new InMemoryClientChannelStorage() });

console.log("🚀 Phase B0 spike — server-side batch-settlement");
console.log(`   Receiver (us)  : ${RECEIVER}`);
console.log(`   Buyer (funded) : ${buyerAccount.address}`);

## 1. Set up the server scheme

Self-managed `receiverAuthorizer` (throwaway key here — in production this is `RECEIVER_AUTHORIZER_PRIVATE_KEY`,
a real secret). `onchainStateTtlMs` is set explicitly short — **this is the #1 finding of this spike**: the
SDK's *default* is `min(5min, max(30s, withdrawDelay/3))`, and since `withdrawDelay` has a protocol floor of
900s (`MIN_WITHDRAW_DELAY`), the default TTL is **always exactly 5 minutes**, regardless of config. Left at
the default, a user's first chat message right after depositing could see a stale cached on-chain balance and
fail with `cumulative_exceeds_balance` for up to 5 minutes. Overriding it here to a few seconds trades a small
amount of extra RPC-read traffic for a much better "deposit → chat immediately" experience.


In [ ]:
const receiverAuthorizerAccount = privateKeyToAccount(generatePrivateKey());
const receiverAuthorizerSigner = {
    address: receiverAuthorizerAccount.address,
    signTypedData: (a: any) => receiverAuthorizerAccount.signTypedData(a),
};
const storage = new InMemoryChannelStorage();
const serverScheme = new ServerScheme(RECEIVER, {
    storage,
    receiverAuthorizerSigner,
    onchainStateTtlMs: 5000, // see finding above — default would be 5 minutes
});

const facilitatorClient = new HTTPFacilitatorClient({ url: FACILITATOR_URL });
const resourceServer = new x402ResourceServer(facilitatorClient);
resourceServer.register(network as any, serverScheme);
console.log("✅ server scheme registered for", network);

## 2. Build payment requirements via the server scheme


In [ ]:
const baseRequirements = {
    scheme: "batch-settlement", network, amount: "4000",
    asset: "0x036CbD53842c5426634e7929541eC2318f3dCF7e",
    payTo: RECEIVER, maxTimeoutSeconds: 3600,
};
const enhanced = await serverScheme.enhancePaymentRequirements(baseRequirements as any, {} as any, []);
console.log(JSON.stringify(enhanced, null, 2));

## 3. Buyer creates the deposit + first voucher payload (off-chain signing only)


In [ ]:
const depositPayload = await buyerScheme.createPaymentPayload(2, enhanced as any);
const channelId = (depositPayload.payload as any).voucher.channelId;
console.log("channelId:", channelId);
console.log("payload.type:", depositPayload.payload.type);

## 4. `verifyPayment(deposit)` — BEFORE any on-chain tx

**Finding:** this checks the payer's *actual USDC wallet balance* (can they afford what they're authorizing?),
not on-chain escrow (there is none yet — that's what settling the deposit creates). With a funded buyer this
succeeds immediately, no transaction needed.

**Finding:** `verifyPayment()` **automatically populates local `ChannelStorage`** as a side effect — the handler
does not need to manually call `mergeRequestContext`/`rememberChannelSnapshot` for this to happen.


In [ ]:
const fullPayload = { x402Version: 2, accepted: enhanced, payload: depositPayload.payload };
const verifyResult = await resourceServer.verifyPayment(fullPayload as any, enhanced as any);
console.log("verifyResult:", JSON.stringify(verifyResult, null, 2));
console.log("\nstorage after verify (before settle):", await storage.get(channelId));

## 5. `settlePayment(deposit)` — the ONE on-chain transaction for this channel's lifetime-so-far

> ⚠️ Real on-chain tx on Base Sepolia.


In [ ]:
const settleResult = await resourceServer.settlePayment(fullPayload as any, enhanced as any);
console.log("settleResult:", JSON.stringify(settleResult, null, 2));
console.log("\nstorage after settle:", await storage.get(channelId));
console.log("\nwaiting 6s for onchainStateTtlMs (5000ms) to expire...");
await new Promise((r) => setTimeout(r, 6000));

## 6. A follow-up chat message — buyer signs a new voucher, server just verifies (NO on-chain tx)

This is the core batching/privacy claim, confirmed empirically: a second (and every subsequent) request on this
channel costs **zero** on-chain transactions — just an off-chain signature + a local `verifyPayment()` call.


In [ ]:
const voucher2 = await signVoucher(buyerSigner, channelId, "8000", network);
const voucherPayload = {
    x402Version: 2, accepted: enhanced,
    payload: { type: "voucher", channelConfig: (depositPayload.payload as any).channelConfig, voucher: voucher2 },
};
const verifyVoucher = await resourceServer.verifyPayment(voucherPayload as any, enhanced as any);
console.log("verifyResult (voucher):", JSON.stringify(verifyVoucher, null, 2));
console.log("\nstorage after verify(voucher):", await storage.get(channelId));

## 7. Charge for the actual LLM cost — the handler's own responsibility

**Finding:** there is no `setSettlementOverrides`-style API for batch-settlement (that's a separate `upto`-scheme
mechanism). The real mechanism is simpler: the resource server owns `ChannelStorage` directly, so after computing
real cost via `convertTokensToCost()` (unchanged from `llm_service.ts`), the handler just increments
`chargedCumulativeAmount` itself, capped at whatever the client has authorized (`signedMaxClaimable`).


In [ ]:
const updateResult = await storage.updateChannel(channelId, (current) => {
    if (!current) return current;
    const realCost = 1234n; // stand-in for convertTokensToCost(actualTokenCount)
    const newCharged = BigInt(current.chargedCumulativeAmount) + realCost;
    return { ...current, chargedCumulativeAmount: newCharged.toString() };
});
console.log("updateChannel status:", updateResult.status);
console.log("storage after charging:", await storage.get(channelId));

## Spike findings (confirmed) — feeding directly into `sc_llm_x402.ts` (Phase B3)

1. ✅ **`verifyPayment()` auto-manages `ChannelStorage`** (creates the record, tracks `pendingRequest`,
   initializes `chargedCumulativeAmount`) — the handler doesn't need to manually orchestrate this for the
   basic verify flow.
2. ✅ **Deposit verification checks the payer's real USDC balance**, not on-chain escrow (there is none yet).
3. ✅ **`settlePayment()` is needed exactly once per channel** (the deposit) — subsequent per-message vouchers
   are verified with **zero on-chain transactions**, confirmed empirically. This is the batching/privacy claim,
   proven, not assumed.
4. 🐛→✅ **`onchainStateTtlMs` defaults to a fixed 5 minutes** (floor-bound by the protocol's
   `MIN_WITHDRAW_DELAY = 900s`), which would otherwise cause the first chat message right after a deposit to
   spuriously fail. `sc_llm_x402.ts` must set this explicitly short (e.g. 5–15s).
5. ✅ **`setSettlementOverrides` does not apply to batch-settlement** (corrects an earlier planning assumption
   — that API is for the separate `upto` scheme). The real "charge actual cost" mechanism is a direct
   `storage.updateChannel()` call bumping `chargedCumulativeAmount`, confirmed working cleanly.

**Next: Phase B1** — extend `@fretchen/s3-utils` with conditional writes, so this same `ChannelStorage`
interface can be backed by S3 instead of `InMemoryChannelStorage`. See `assistent_plan.md` §F.
